# A3 — Resonance-Based Router (RBR)

ART tarzı memory bank + vigilance eşiği. Tokenlar prototype slot'larla rezonansa girer; eşik altı tokenlar yeni slot açar.

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/A3_RBR_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
PARAMS = {
    "method_name":   "A3_RBR",
    "run_name":      "wt2_seed42",
    "seed":          42,
    "results_root":  "./results",

    "model_name":    "gpt2",
    "tokenizer_name": "gpt2",

    "dataset":       "wikitext2",
    "max_seq_len":   512,
    "batch_size":    16,
    "num_workers":   2,

    "learning_rate": 5e-5,
    "num_epochs":    3,
    "warmup_steps":  100,
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    "limit_train_batches": 100,
    "limit_eval_batches":  50,

    "log_every_n_steps":        25,
    "save_checkpoints":         False,
    "checkpoint_every_n_steps": 500,
    "keep_last_n_checkpoints":  3,

    # ─ Method-specific (RBR)
    "rbr_vigilance":  0.7,        # eşik θ
    "rbr_num_slots":  64,         # bellek slot sayısı
    "rbr_slot_dim":   768,        # = hidden_size
    "rbr_beta":       1e-6,       # stabilite
    "rbr_slot_lr":    0.1,        # slot güncelleme hızı
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method — Resonance-Based Router ──────────────────────────
class ResonanceBasedRouter(BaseRouterAttention):
    def __init__(self, gpt2_config, method_params):
        super().__init__(gpt2_config, method_params)
        mp = method_params
        self.vigilance = float(mp.get("rbr_vigilance", 0.7))
        self.num_slots = int(mp.get("rbr_num_slots", 64))
        self.slot_dim  = int(mp.get("rbr_slot_dim", self.embed_dim))
        self.beta      = float(mp.get("rbr_beta", 1e-6))
        self.slot_lr   = float(mp.get("rbr_slot_lr", 0.1))
        self.memory_bank = nn.Parameter(torch.randn(self.num_slots, self.slot_dim) * 0.02)
        self.register_buffer("slot_usage", torch.zeros(self.num_slots))
        self.value_proj = nn.Linear(self.slot_dim, self.embed_dim)

    def _resonance(self, hidden):
        m_norm = self.memory_bank / (self.memory_bank.norm(dim=-1, keepdim=True) + self.beta)
        return torch.einsum("bsh,kh->bsk", hidden, m_norm)

    def _decide(self, res):
        mask = res >= self.vigilance
        masked = res * mask.float()
        row_sum = masked.sum(-1, keepdim=True).clamp(min=1e-8)
        weights = masked / row_sum
        mismatch = mask.sum(-1) == 0
        return weights, mismatch

    def _maybe_open_slot(self, hidden, mismatch):
        if not self.training or not mismatch.any():
            return
        with torch.no_grad():
            new_proto = hidden[mismatch].mean(0)
            target = int(self.slot_usage.argmin().item())
            self.memory_bank.data[target] = (
                (1 - self.slot_lr) * self.memory_bank.data[target]
                + self.slot_lr * new_proto
            )
            self.slot_usage[target] = 0
            self.slot_usage += 1

    def _attend(self, q, k, v, hidden_states, attention_mask):
        res = self._resonance(hidden_states)
        weights, mismatch = self._decide(res)
        self._maybe_open_slot(hidden_states.detach(), mismatch)
        mem_out = torch.einsum("bsk,kh->bsh", weights, self.memory_bank)
        out = self.value_proj(mem_out)
        out = self._split_heads(out)
        self._last_stats = {"mismatch_rate": float(mismatch.float().mean().item())}
        return out, weights.unsqueeze(1)

print("ResonanceBasedRouter tanımlandı.")

In [ ]:
def build_model_and_adapter(params):
    wrapper = GPT2Wrapper(params["model_name"])
    wrapper.inject_attention(ResonanceBasedRouter, params)
    print(f"RBR injected → {sum(p.numel() for p in wrapper.parameters()):,} params")
    return wrapper, None

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
fm = result["final_metrics"]
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL:             {fm['test_ppl']:.4f}")
print(f"Final test BPC (per-char):  {fm['test_bpc']:.4f}")
print(f"Final test bits-per-token:  {fm['test_bits_per_token']:.4f}")
print(f"chars/token:                {fm['chars_per_token']:.3f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")